# 12 — Synthèse consolidée : Système A + Système B dédupliqué

**Objectif** — la vue la plus complète que ces deux systèmes permettent
ensemble : volumes, sous-motifs, canaux, agences, exposition financière,
recommandations — sur le périmètre le plus large possible, sans compter un
ticket deux fois (notebook 09).

| | |
|---|---|
| **Entrée** | Système A (notebooks 01-07), Système B dédupliqué (notebooks 08-10), validation croisée (notebook 11) |
| **Sorties** | `resultats/tables/12_*.csv` + figures |
| **Suite** | rédaction du mémoire |

**Ce que cette synthèse ajoute par rapport à `niv/notebooks/06`** (la
synthèse initiale, Système A seul) : la dimension agence, un sous-motif
annoté à la main sur 80 % du Système B, une mesure réelle de résolution, et
une exposition financière mieux couverte. Elle ne remplace pas le notebook
06 — elle l'étend avec ce que le Système B apporte, en gardant les mêmes
réserves de fond (§ 6.1 du protocole : toujours aucun journal de
transactions, toujours aucun dénominateur).

In [1]:
import sys
from pathlib import Path

RACINE = Path.cwd().parent
sys.path.insert(0, str(RACINE / "src"))

import matplotlib.pyplot as plt
import pandas as pd

from reclamations import chargement, config, consolidation, systeme_b, texte, viz

viz.appliquer_charte()
pd.set_option("display.width", 160)

# --- Système A : périmètre opérationnel complet + classification causale ---
df_a = chargement.typer(chargement.charger_brut())
tickets_a = chargement.perimetre_operationnel(df_a)
messages = chargement.charger_messages_ouverture()
tickets_a = tickets_a.copy()
tickets_a["texte"] = chargement.construire_texte_enrichi(tickets_a, messages)
tickets_a["a_texte"] = texte.a_texte_exploitable(tickets_a["texte"])
tickets_a["famille"] = texte.classer(tickets_a["texte"])

# --- Système B : dédupliqué (notebook 09) ---
tickets_b_brut = systeme_b.charger_tickets()
categories_b = systeme_b.charger_categories()
groups_b = systeme_b.charger_groups()
champs_b, valeurs_b = systeme_b.charger_champs_personnalises()
sous_motifs_b = systeme_b.sous_motifs(champs_b, valeurs_b)

cles_a = consolidation.cles_a(tickets_a)
cles_b = consolidation.cles_b(tickets_b_brut, champs_b, valeurs_b)
paires = consolidation.apparier(cles_a, cles_b)
tickets_b = consolidation.tickets_b_non_apparies(tickets_b_brut, paires)

print(f"Système A (période opérationnelle) : {len(tickets_a):,}")
print(f"Système B (dédupliqué)             : {len(tickets_b):,}")
print(f"TOTAL CONSOLIDÉ                    : {len(tickets_a) + len(tickets_b):,}")

/home/gauss/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


/home/gauss/Bureau/Afriland/analyse des reclamation clients/niv/src/reclamations/texte.py:261: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  correspond = texte.str.contains(motif, regex=True, na=False)
/home/gauss/Bureau/Afriland/analyse des reclamation clients/niv/src/reclamations/texte.py:261: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  correspond = texte.str.contains(motif, regex=True, na=False)


/home/gauss/Bureau/Afriland/analyse des reclamation clients/niv/src/reclamations/texte.py:261: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  correspond = texte.str.contains(motif, regex=True, na=False)
/home/gauss/Bureau/Afriland/analyse des reclamation clients/niv/src/reclamations/texte.py:261: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  correspond = texte.str.contains(motif, regex=True, na=False)
/home/gauss/Bureau/Afriland/analyse des reclamation clients/niv/src/reclamations/texte.py:261: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  correspond = texte.str.contains(motif, regex=True, na=False)


/home/gauss/Bureau/Afriland/analyse des reclamation clients/niv/src/reclamations/texte.py:261: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  correspond = texte.str.contains(motif, regex=True, na=False)


Système A (période opérationnelle) : 18,056
Système B (dédupliqué)             : 6,823
TOTAL CONSOLIDÉ                    : 24,879


## 1. Les deux taxonomies, côte à côte — pas de mapping forcé

`texte.FAMILLES` (A, règles) et les sous-motifs du Système B (annotation
humaine) restent deux nomenclatures différentes. Le notebook 11 a validé la
correspondance sur deux familles seulement (`erreur_client` ↔ `Bénéficiaire
erroné`, précision 0,93/rappel 0,60 ; `debit_non_credit` ↔ les sous-motifs
« échec de transaction », précision 0,93/rappel 0,49) — c'est le seul pont
de confiance disponible entre les deux tables ci-dessous.

In [2]:
base_a = tickets_a[tickets_a["a_texte"]]
familles_a = texte.couverture(base_a["famille"])
chargement.sauver_table(familles_a, "12_familles_causales_a")

sm_b = sous_motifs_b.loc[sous_motifs_b.index.isin(tickets_b["id"])]
resume_sm_b = (
    sm_b.groupby(["domaine", "sous_motif"]).size().rename("effectif").reset_index()
    .sort_values("effectif", ascending=False)
)
chargement.sauver_table(resume_sm_b, "12_sous_motifs_b", index=False)

print("Système A -- familles causales (base textuelle) :")
print(familles_a[["libelle", "effectif", "pct"]])
print()
print("Système B -- 10 sous-motifs les plus fréquents :")
print(resume_sm_b.head(10).to_string(index=False))

  -> resultats/tables/12_familles_causales_a.csv
  -> resultats/tables/12_sous_motifs_b.csv
Système A -- familles causales (base textuelle) :
                                                            libelle  effectif   pct
famille                                                                            
non_classe                                Non classé par les règles      4759  46.0
debit_non_credit       Débité sans que le bénéficiaire soit crédité      3830  37.0
erreur_client     Erreur de saisie du client (mauvais bénéficiaire)       643   6.2
acces_otp                     Accès bloqué / OTP / authentification       496   4.8
debit_injustifie       Débit injustifié, doublé, ou frais contestés       312   3.0
carte                                 Carte bancaire / distributeur       230   2.2
demande_info            Demande d'information (pas une réclamation)        84   0.8

Système B -- 10 sous-motifs les plus fréquents :
                         domaine                   s

## 2. Canal — deux vocabulaires différents, non fusionnés

`channel` (A) décrit le sous-canal digital d'Intercom (android, iOS,
whatsapp...). `canal` (B) décrit le canal d'intake au sens large, où
`INTERCOM` n'est qu'une valeur parmi d'autres (agence, courrier, appel...).
Les fusionner donnerait une fausse impression de compatibilité — présentés
côte à côte.

In [3]:
canal_a = tickets_a["channel"].value_counts().rename("effectif").to_frame()
canal_a["pct"] = (canal_a["effectif"] / len(tickets_a) * 100).round(1)
canal_b = tickets_b["canal"].value_counts().rename("effectif").to_frame()
canal_b["pct"] = (canal_b["effectif"] / len(tickets_b) * 100).round(1)

chargement.sauver_table(canal_a, "12_canal_a")
chargement.sauver_table(canal_b, "12_canal_b")

print("Système A -- channel :")
print(canal_a)
print()
print("Système B (dédupliqué) -- canal :")
print(canal_b)

  -> resultats/tables/12_canal_a.csv
  -> resultats/tables/12_canal_b.csv
Système A -- channel :
           effectif   pct
channel                  
android       11857  65.7
ios            3711  20.6
whatsapp       2297  12.7
facebook        181   1.0
messenger         7   0.0
instagram         3   0.0

Système B (dédupliqué) -- canal :
           effectif   pct
canal                    
INTERCOM       2291  33.6
EN_AGENCE      1970  28.9
COURRIER        995  14.6
APPEL           470   6.9
COLLEGUE        322   4.7
MAIL            286   4.2
WHATSAPP        243   3.6
AUTRE           241   3.5


## 3. Agence — une dimension que seul le Système B apporte

In [4]:
noms_groupes = tickets_b["source_group_id"].map(groups_b.set_index("id")["group_name"])
est_agence = noms_groupes.str.startswith("FIRST_BANK_", na=False)
agences = noms_groupes[est_agence].value_counts().rename("effectif").to_frame().head(10)
chargement.sauver_table(agences, "12_top_agences")
agences

  -> resultats/tables/12_top_agences.csv


,effectif
source_group_id,
FIRST_BANK_HIPPODROME,255
FIRST_BANK_NGAOUNDERE,254
FIRST_BANK_MESSAMENDONGO,107
FIRST_BANK_KRIBI,101
FIRST_BANK_FENETRE_ISLAMIQUE_YAOUNDE,100
FIRST_BANK_DSCHANG,92
FIRST_BANK_MAROUA,85
FIRST_BANK_BONAMOUSSADI,80
FIRST_BANK_NDOKOTTI,77


## 4. Exposition financière — comparée sur les deux familles validées (notebook 11)

Le Système B couvre le montant sur 92,7 % des tickets contre 39,96 % côté A
(protocole § 3.4) : plus fiable en couverture, mais soumis aux mêmes réserves
(ordre de grandeur, jamais un chiffre officiel).

In [5]:
# Mapping valide au notebook 11 -- pas devine ici.
SOUS_MOTIFS_ECHEC_TRANSACTION = {
    "Échec Cash-In", "Échec Cash-Out", "Échec Wallet-to-Wallet",
    "Échec Wallet-to-Other-Wallet", "Fond non perçu",
    "Échec Achat Air-Time", "Échec MAC Bank-to-Wallet", "Échec MAC Wallet-to-Bank",
}

mt_a = chargement.montants_plausibles(base_a)
expo_a = mt_a[mt_a["famille"].isin(["debit_non_credit", "erreur_client"])].groupby("famille")["montant"].agg(
    tickets="count", mediane="median"
)

mt_b = systeme_b.montants_plausibles(champs_b, valeurs_b)
mt_b = mt_b.loc[mt_b.index.isin(tickets_b["id"])]
sm_b_montant = sm_b.join(mt_b.rename("montant"), how="inner")
sm_b_montant["famille_equivalente"] = sm_b_montant["sous_motif"].map(
    lambda s: "debit_non_credit" if s in SOUS_MOTIFS_ECHEC_TRANSACTION else ("erreur_client" if s == "Bénéficiaire erroné" else None)
)
expo_b = sm_b_montant.dropna(subset=["famille_equivalente"]).groupby("famille_equivalente")["montant"].agg(
    tickets="count", mediane="median"
)

comparaison = pd.DataFrame({"A (règles)": expo_a["mediane"], "B (annotation)": expo_b["mediane"]}).round(0)
comparaison["A_tickets"] = expo_a["tickets"]
comparaison["B_tickets"] = expo_b["tickets"]
chargement.sauver_table(comparaison, "12_exposition_comparee")
comparaison

  -> resultats/tables/12_exposition_comparee.csv


,A (règles),B (annotation),A_tickets,B_tickets
debit_non_credit,50000.0,50000.0,3217,2886
erreur_client,50000.0,48000.0,277,839


**Convergence notable** : la médiane `debit_non_credit` est identique des
deux côtés (50 000 XAF, sur 3 217 tickets côté A et 2 886 côté B — deux
mesures indépendantes, deux systèmes différents). `erreur_client` est
proche (50 000 contre 48 000). Ce n'est pas une preuve que le montant est
fiable en absolu (protocole § 8.4 point 6, toujours ouvert), mais c'est un
signal de cohérence entre deux sources indépendantes qui n'existait pas
avant ce chantier.

## 5. Recommandations — mises à jour avec ce que le Système B apporte

| Problème | Source | Ampleur | Responsable présumé | Action |
|---|---|---|---|---|
| Débité sans que le bénéficiaire soit crédité | A + B (concordants) | Famille dominante des deux côtés (§ 1) | Système (banque ↔ opérateur) | Diagnostic technique de la chaîne SARA ↔ mobile money (toujours hors de portée, protocole § 6.1) |
| Erreur de saisie du bénéficiaire | A + B (concordants) | 2e famille des deux côtés | UX de l'application | Afficher le nom du titulaire avant validation (déjà recommandé au notebook 04) |
| Rappel des règles causales plafonné (~49-60 %, notebook 11) | A, mesuré via B | 40-50 % des tickets debit_non_credit/erreur_client non captés par les règles | Équipe data | Classification apprise sur le jeu de référence de fait (notebook 11) |
| Concentration géographique des réclamations | B seul | Agences en tête, § 3 | Réseau d'agences concernées | Investigation ciblée sur les agences en tête de la table § 3 — nouveau, impossible avec A seul |
| Délai de traitement | B seul | médiane mesurée (notebook 10 § 5), non mesurable côté A (protocole § 5.3) | Responsable du support | Suivre ce délai comme KPI, en s'appuyant sur B où A ne le permet pas |

Ce tableau ne remplace pas celui du notebook 04 (§ point d'action isolé) — il
y ajoute les deux lignes que seul le Système B rend possibles.

## 6. Ce que la consolidation A+B ne change pas

Reprise du protocole § 6.1, toujours vraie après ce chantier :

1. **Toujours aucun journal de transactions.** Le dénominateur reste absent
   des deux systèmes — le taux d'incident par transaction (protocole § 1.2)
   reste hors de portée.
2. **`type_reclamation` / `is_reclamation_fondee` vides à 100 % dans B**
   (notebook 08 § 3) — la question « réclamation fondée ou non » reste sans
   réponse, malgré le second système.
3. **Le dédoublonnage sous-estime probablement les vrais doublons**
   (notebook 09 § 6) — le total consolidé de ce notebook est un plancher, pas
   un chiffre définitif.
4. **La correspondance de taxonomie (§ 1, § 4 de ce notebook) ne couvre que
   2 des 6 familles causales de A** — `acces_otp`, `debit_injustifie`,
   `carte`, `demande_info` n'ont pas de pont validé vers le Système B
   (notebook 11 § 3).
5. **Les 27 644 conversations Intercom orphelines** restent hors du
   périmètre de comptage (décision actée en amont de ce chantier) — export
   partiel à deux fenêtres, aucune n'a de `ticket_state`.

**Prochaine étape** — les points bloquants du protocole § 8.4 restent à
lever avec le métier (mode d'extraction, cadre réglementaire, fiabilité du
champ montant) ; ce chantier ne les a pas résolus, il a élargi ce qui est
mesurable en attendant.